# Prototype 1: PDF → Markdown ingestion

Parses every PDF in `data/raw/` with MinerU and saves one Markdown file per PDF in `data/processed/markdown/`.

Flow: **config → list PDFs → parse + export → verify every page made it.**

Re-running is safe: a PDF is skipped only if its Markdown already contains *every* page.

In [21]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from mineru import DoclibClient
from mineru.doclib import ParseRequest
from mineru.doclib.types import DocContentExportRequest

# ---- config ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MARKDOWN_DIR = PROJECT_ROOT / "data" / "processed" / "markdown"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

TIER = "flash"      # one tier everywhere: flash | basic | standard | advanced
PAGE_RANGE = "all"     # IMPORTANT: MinerU parses only pages 1-10 if you don't say "all"
POLL_SECONDS = 5
TIMEOUT_SECONDS = 60 * 60   # per PDF; big PDFs can take a while

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

client = DoclibClient()

In [22]:
pdf_files = sorted(RAW_DIR.glob("*.pdf"))
assert pdf_files, f"No PDFs found in {RAW_DIR}"

display(pd.DataFrame([
    {"filename": p.name, "size_mb": round(p.stat().st_size / 1024**2, 2)}
    for p in pdf_files
]))

,filename,size_mb
0,AGI.pdf,7.03
1,Foundation_Models.pdf,13.75
2,GPT-4.pdf,5.00
3,Holistic_Eval.pdf,5.77
4,LLMs.pdf,5.58


## Helpers

- `pages_in_markdown` reads the `<!-- page X of N -->` markers MinerU writes, so we can tell if a file is complete.
- `parse_pdf` asks for **all** pages, waits (with a timeout), then exports the whole document in one call.

In [23]:
PAGE_MARKER = re.compile(r"<!-- page (\d+) of (\d+) -->")


def pages_in_markdown(md_path):
    """Return (pages_found, total_pages) from MinerU page markers. (0, None) if unknown."""
    if not md_path.exists():
        return 0, None
    markers = PAGE_MARKER.findall(md_path.read_text(encoding="utf-8"))
    if not markers:
        return 0, None
    pages = {int(page) for page, _ in markers}
    total = int(markers[0][1])
    return len(pages), total


def is_complete(md_path):
    found, total = pages_in_markdown(md_path)
    return total is not None and found == total


def wait_for_parse(parse_id):
    deadline = time.time() + TIMEOUT_SECONDS
    parse = client.get_parse(parse_id)
    while parse.status in ("pending", "parsing"):
        if time.time() > deadline:
            raise TimeoutError(f"Parse {parse_id} still '{parse.status}' after {TIMEOUT_SECONDS}s")
        time.sleep(POLL_SECONDS)
        parse = client.get_parse(parse_id)
    if parse.status != "done":
        raise RuntimeError(f"Parse {parse_id} ended as {parse.status}: {parse.error_code} {parse.error_msg}")


def parse_pdf(pdf_path):
    md_path = MARKDOWN_DIR / f"{pdf_path.stem}.md"

    submit = client.ensure_parse(ParseRequest(
        path=str(pdf_path.resolve()),
        tier=TIER,
        page_range=PAGE_RANGE,
    ))
    for parse_id in submit.wait_parse_ids:
        wait_for_parse(parse_id)

    # Export the whole document straight to a .md file (no page-by-page loop needed)
    doc_ref = submit.short_id
    if doc_ref is None:
        raise ValueError(f"Missing short_id for {pdf_path.name}; parse response: {submit}")
    client.export_doc_content(
        doc_ref,
        DocContentExportRequest(
            tier=TIER,
            page_range=PAGE_RANGE,
            output=str(md_path.resolve()),
        ),
    )

    found, total = pages_in_markdown(md_path)
    return {
        "document": pdf_path.name,
        "status": "success" if found == total else "incomplete",
        "pages": f"{found}/{total}",
        "characters": md_path.stat().st_size,
        "markdown_path": str(md_path),
    }

## Parse every PDF

Failures don't stop the loop; they're recorded in `artifacts/ingestion_manifest.json`.

In [24]:
manifest = []

for pdf in tqdm(pdf_files, desc="Parsing PDFs"):
    md_path = MARKDOWN_DIR / f"{pdf.stem}.md"

    if is_complete(md_path):
        found, total = pages_in_markdown(md_path)
        manifest.append({"document": pdf.name, "status": "skipped_complete",
                         "pages": f"{found}/{total}", "markdown_path": str(md_path)})
        continue

    try:
        manifest.append(parse_pdf(pdf))
    except Exception as exc:
        manifest.append({"document": pdf.name, "status": "failed", "error": str(exc)})

    # save after every PDF so a crash doesn't lose progress
    (ARTIFACTS_DIR / "ingestion_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(pd.DataFrame(manifest))

Parsing PDFs:   0%|          | 0/5 [00:00<?, ?it/s]

,document,status,pages,characters,markdown_path
0,AGI.pdf,success,155/155,538169,c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\d...
1,Foundation_Models.pdf,success,214/214,917015,c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\d...
2,GPT-4.pdf,success,100/100,309957,c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\d...
3,Holistic_Eval.pdf,success,162/162,700387,c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\d...
4,LLMs.pdf,success,144/144,916121,c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\d...


## Verify: every raw PDF has a complete Markdown file

In [25]:
report = []
for pdf in pdf_files:
    md_path = MARKDOWN_DIR / f"{pdf.stem}.md"
    found, total = pages_in_markdown(md_path)
    text = md_path.read_text(encoding="utf-8") if md_path.exists() else ""
    report.append({
        "pdf": pdf.name,
        "markdown_exists": md_path.exists(),
        "pages_found": found,
        "pages_total": total,
        "complete": total is not None and found == total,
        "words": len(text.split()),
        "headings": sum(line.lstrip().startswith("#") for line in text.splitlines()),
        "preview": " ".join(text.split())[:150],
    })

report_df = pd.DataFrame(report)
display(report_df)

missing = report_df.loc[~report_df["complete"], "pdf"].tolist()
assert not missing, f"Not fully parsed: {missing}. Re-run the parse cell (or check the manifest for errors)."
print(f"All {len(pdf_files)} PDFs fully parsed ✔")

,pdf,markdown_exists,pages_found,pages_total,complete,words,headings,preview
0,AGI.pdf,True,155,155,True,89506,327,<!-- page 1 of 155 --> # Sparks of Artificial ...
1,Foundation_Models.pdf,True,214,214,True,123706,146,<!-- page 1 of 214 --> arXiv:2108.07258v3 [cs....
2,GPT-4.pdf,True,100,100,True,46063,191,<!-- page 1 of 100 --> arXiv:2303.08774v6 [cs....
3,Holistic_Eval.pdf,True,162,162,True,97994,186,<!-- page 1 of 162 --> Published in Transactio...
4,LLMs.pdf,True,144,144,True,131998,129,<!-- page 1 of 144 --> 1 # A Survey of Large L...


All 5 PDFs fully parsed ✔


# Part 2: Plain RAG baseline (clean → sections → chunks → index → retrieve → rerank)

```
markdown ─► clean ─► structure-aware sections ─► semantic chunks ─► chunks.jsonl
                                                                      ├─► Chroma (dense)
                                                                      └─► BM25 (sparse)
query ─► dense top-30 + BM25 top-30 ─► RRF ─► top-20 ─► cross-encoder rerank ─► top-5
```

`chunks.jsonl` is the source of truth. Every chunk has a stable `chunk_id`; the GraphRAG part will point back to these same IDs.

Install once: `pip install -U langchain-huggingface langchain-chroma chromadb rank_bm25 sentence-transformers`

In [ ]:
import hashlib
from collections import Counter

import numpy as np
import torch
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ---- paths ----
CLEAN_DIR = PROJECT_ROOT / "data" / "processed" / "clean"
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.jsonl"
CHROMA_DIR = PROJECT_ROOT / "data" / "index" / "chroma"
COLLECTION = "okf_chunks"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ---- models ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
RERANK_MODEL = "BAAI/bge-reranker-base"
BGE_QUERY_PROMPT = "Represent this sentence for searching relevant passages: "

# ---- chunking ----
MAX_WORDS = 300          # bge max is 512 tokens; 300 words + header stays under it
MIN_WORDS = 80           # don't cut a chunk for a topic shift before it has this many words
TINY_WORDS = 40          # chunks smaller than this get merged into the previous one
BREAK_PERCENTILE = 20    # topic shift = paragraph similarity in the lowest 20%
REBUILD_CHUNKS = False   # True = re-clean + re-chunk even if chunks.jsonl exists

# ---- retrieval ----
DENSE_K = 30
BM25_K = 30
RRF_K = 60               # standard RRF constant
RERANK_TOP_N = 20        # how many fused results go to the reranker
FINAL_K = 5

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    # bge wants this instruction on queries only (not on documents)
    query_encode_kwargs={"normalize_embeddings": True, "prompt": BGE_QUERY_PROMPT},
)
print("device:", DEVICE)

## Step 1: Cleaning

What the MinerU (flash) output contains that we don't want:
- image placeholders `![Formula block](doc:...)` and `<details>/<summary>image content</summary>` wrappers
- running page headers repeated on every page, e.g. *"Published in Transactions on Machine Learning Research"*
- page numbers and chart axis labels (`95`, `0.4`, `0.2`), which are lines that are only numbers
- footnote markers `<sup>11</sup>`

Page markers are kept for now because the next step needs them for page numbers.
Cleaned files are saved to `data/processed/clean/` so you can open and inspect them.

In [ ]:
NOISE_LINE_RE = re.compile(
    r"^("
    r"!\[.*\]\(.*\)"                 # image / formula placeholders
    r"|</?details>"
    r"|<summary>.*</summary>"
    r"|[\d\s.,%()\-–]+"               # only numbers: page numbers, chart axes
    r")$"
)
SUP_RE = re.compile(r"<sup>.*?</sup>")


def find_running_headers(lines, n_pages, min_share=0.3):
    """Lines repeated on >=30% of pages are page headers/footers, not content."""
    candidates = (
        l.strip() for l in lines
        if 15 < len(l.strip()) < 120 and not l.strip().startswith(("|", "```", "<!--", "#"))
    )
    counts = Counter(candidates)
    return {line for line, n in counts.items() if n >= max(5, min_share * n_pages)}


def clean_markdown(text):
    lines = text.splitlines()
    _, n_pages = PAGE_MARKER.findall(text)[0] if PAGE_MARKER.search(text) else (0, 1)
    headers = find_running_headers(lines, int(n_pages))

    kept, removed = [], 0
    for line in lines:
        s = line.strip()
        if s.startswith("<!-- Source page"):            # old marker from the first notebook version
            removed += 1
            continue
        if s and not s.startswith("<!--") and (NOISE_LINE_RE.match(s) or s in headers):
            removed += 1
            continue
        kept.append(SUP_RE.sub("", line))

    cleaned = re.sub(r"\n{3,}", "\n\n", "\n".join(kept))   # collapse big gaps
    return cleaned, {"lines_removed": removed, "running_headers": sorted(headers)}


cleaning_report = []
for md_path in sorted(MARKDOWN_DIR.glob("*.md")):
    cleaned, info = clean_markdown(md_path.read_text(encoding="utf-8"))
    (CLEAN_DIR / md_path.name).write_text(cleaned, encoding="utf-8")
    cleaning_report.append({"doc": md_path.stem, **info,
                            "chars_before": md_path.stat().st_size, "chars_after": len(cleaned)})

display(pd.DataFrame(cleaning_report))

## Step 2: Structure-aware sections

MinerU flattened every heading to `##`, so the `#` count is useless for hierarchy. The **section number** gives it instead:
`3` → depth 1, `3.1` → depth 2, `B.2.1` → depth 3.

Rules:
- **Numbered heading** (`3.1 Law`, `A.6 Codeforces rating`): a real section. Update the section path.
- **Named heading** (Abstract, Introduction, Conclusion, Appendix…): a top-level section.
- **Any other `##`**: it's not a heading. MinerU marked dialogue lines like `## GPT-4:` or `## Juliet: ...` as headings, so these become normal text.
- **Contents / References**: skipped. A reference list is thousands of author names that pollute BM25 and the graph.

Each block keeps `doc`, `section` and `page`, the same idea as your `extract_blocks`.

In [ ]:
# "3.1 Law", "B.2.1 MS MARCO", "A GPT-4 has common sense grounding"
NUMBERED_RE = re.compile(r"^(?P<num>(?:\d{1,2}|[A-H])(?:\.\d{1,2})*)\.?\s+(?P<title>[A-Z][^\n]{1,150})$")
NAMED_SECTIONS = {"abstract", "introduction", "conclusion", "conclusions", "discussion",
                  "acknowledgments", "acknowledgements", "appendix", "contents",
                  "references", "bibliography"}
SKIP_SECTIONS = {"contents", "references", "bibliography"}


def as_section_heading(text):
    """Return (number, label) if `text` is a real section heading, else None.
    number is e.g. "3.1" for numbered sections, None for named ones like "Abstract"."""
    text = re.sub(r"\s+", " ", text).strip()
    if text.lower() in NAMED_SECTIONS:
        return None, text.title()
    m = NUMBERED_RE.match(text)
    if m and "," not in text:            # commas = author lists / citations, not headings
        return m["num"], f"{m['num']} {m['title']}"
    return None


def update_path(path, number, label):
    """path is a list of (number, label). Keep only ancestors of the new section.
    Uses the numbers (not position), so a missing "3" heading can't leave "3.1" under "2"."""
    if number is None:
        return [(None, label)]
    parents = [(n, l) for n, l in path if n is not None and number.startswith(n + ".")]
    return parents + [(number, label)]


def extract_sections(cleaned_text, doc):
    """Split cleaned markdown into blocks (paragraph / table / code) with section path + page."""
    title, path, page = doc, [], None
    blocks, buf, in_code = [], [], False

    def in_skip_section():
        return bool(path) and path[0][1].split()[-1].lower() in SKIP_SECTIONS

    def flush():
        nonlocal buf
        text = "\n".join(buf).strip()
        buf = []
        if text and not in_skip_section():
            blocks.append({"doc": doc, "section": " > ".join(l for _, l in path) or "Front matter",
                           "page": page, "text": text})

    for line in cleaned_text.splitlines():
        s = line.strip()

        if s.startswith("```"):                       # keep code blocks in one piece
            in_code = not in_code
            buf.append(line)
            continue
        if in_code:
            buf.append(line)
            continue

        if m := PAGE_MARKER.match(s):
            page = int(m.group(1))
            continue

        is_md_heading = s.startswith("#")
        heading_text = s.lstrip("#").strip()

        # Plain-text "References" / "A Appendix title" lines also count (AGI.md has them unmarked),
        # but only short lines without a trailing ":", so sentences are never mistaken for headings.
        if is_md_heading:
            heading = as_section_heading(heading_text.rstrip(":"))
        elif len(s) < 80 and not s.endswith(":"):
            heading = as_section_heading(s)
            # plain lines may only *start* references or *end* a skipped section
            if heading and not (heading[1].lower() in SKIP_SECTIONS or in_skip_section()):
                heading = None
        else:
            heading = None

        # first heading before any section = paper title ("# A Survey..." would otherwise look like appendix A)
        if is_md_heading and title == doc and not path and (s.startswith("# ") or heading is None):
            flush()
            title = heading_text
            continue

        if heading:
            flush()
            path = update_path(path, *heading)
            continue

        if not s:
            flush()
        else:
            buf.append(heading_text if is_md_heading else line)   # demoted "##" lines become text

    flush()
    return title, blocks


doc_titles, doc_blocks = {}, {}
for clean_path in sorted(CLEAN_DIR.glob("*.md")):
    title, blocks = extract_sections(clean_path.read_text(encoding="utf-8"), clean_path.stem)
    doc_titles[clean_path.stem], doc_blocks[clean_path.stem] = title, blocks

display(pd.DataFrame([
    {"doc": d, "title": doc_titles[d][:60], "blocks": len(b),
     "sections": len({x["section"] for x in b}), "words": sum(len(x["text"].split()) for x in b)}
    for d, b in doc_blocks.items()
]))

# peek at the section tree of one doc
print(*list(dict.fromkeys(b["section"] for b in doc_blocks["Holistic_Eval"]))[:15], sep="\n")

## Step 3: Semantic chunking

Within each section:
1. Blocks bigger than `MAX_WORDS` are split first. Tables are split **by rows, repeating the header row**, so every piece is still a readable table.
2. Each block is embedded, and we measure the cosine similarity between neighbouring blocks.
3. Blocks are packed into a chunk until one of these happens:
   - the **section changes** (chunks never cross sections)
   - the chunk would exceed `MAX_WORDS`
   - there's a **topic shift**: similarity drops into the lowest `BREAK_PERCENTILE`%. This is the "semantic" part.
4. Tiny leftover chunks are merged into the previous chunk of the same section.

It works on paragraphs rather than single sentences, which keeps it fast: about 10k embeddings instead of about 100k.

The chunk text gets a header line `Paper title > section`. That helps both BM25 and the embeddings know what the chunk is about.

In [ ]:
OVERLAP_WORDS = 50


def word_windows(text, max_words=MAX_WORDS):
    """Sliding window over words, with overlap so a sentence cut at the edge appears in both."""
    words = text.split()
    step = max_words - OVERLAP_WORDS
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), step)
            if words[i:i + max_words]]


def split_big_block(text, max_words=MAX_WORDS):
    if len(text.split()) <= max_words:
        return [text]

    lines = text.splitlines()
    if lines[0].lstrip().startswith("|") and len(lines) > 2:     # markdown table
        header, rows, pieces, cur = lines[:2], lines[2:], [], []
        for row in rows:
            if cur and len(" ".join(header + cur + [row]).split()) > max_words:
                pieces.append("\n".join(header + cur))
                cur = []
            cur.append(row)
        if cur:
            pieces.append("\n".join(header + cur))
        # a single giant row (broken table) still gets cut down
        return [p for piece in pieces
                for p in (word_windows(piece) if len(piece.split()) > max_words else [piece])]

    return word_windows(text, max_words)                         # prose


def semantic_chunks(blocks):
    units = [{**b, "text": piece} for b in blocks for piece in split_big_block(b["text"])]
    if len(units) < 2:
        return [units] if units else []

    vecs = np.array(embeddings.embed_documents([u["text"] for u in units]))
    sims = (vecs[1:] * vecs[:-1]).sum(axis=1)          # cosine, vectors are normalized
    threshold = np.percentile(sims, BREAK_PERCENTILE)

    groups, cur = [], [units[0]]
    for i, u in enumerate(units[1:], start=1):
        cur_words = sum(len(x["text"].split()) for x in cur)
        new_section = u["section"] != cur[-1]["section"]
        too_big = cur_words + len(u["text"].split()) > MAX_WORDS
        topic_shift = sims[i - 1] < threshold and cur_words >= MIN_WORDS
        if new_section or too_big or topic_shift:
            groups.append(cur)
            cur = [u]
        else:
            cur.append(u)
    groups.append(cur)

    merged = []                                         # merge tiny leftovers
    for g in groups:
        words = sum(len(x["text"].split()) for x in g)
        if merged and words < TINY_WORDS and merged[-1][-1]["section"] == g[0]["section"]:
            merged[-1].extend(g)
        else:
            merged.append(g)
    return merged


def to_document(group, doc, index):
    text = "\n\n".join(u["text"] for u in group)
    section = group[0]["section"]
    pages = [u["page"] for u in group if u["page"] is not None]
    chunk_id = f"{doc}_{index:04d}_{hashlib.sha1(text.encode()).hexdigest()[:8]}"
    return Document(
        page_content=f"{doc_titles[doc]} > {section}\n\n{text}",
        metadata={
            "chunk_id": chunk_id, "doc": doc, "title": doc_titles[doc], "section": section,
            "page_start": min(pages, default=-1), "page_end": max(pages, default=-1),
            "n_words": len(text.split()),
        },
    )


if CHUNKS_PATH.exists() and not REBUILD_CHUNKS:
    with CHUNKS_PATH.open(encoding="utf-8") as f:
        chunk_docs = [Document(**json.loads(line)) for line in f]
    print(f"Loaded {len(chunk_docs)} chunks from {CHUNKS_PATH.name} (set REBUILD_CHUNKS=True to redo)")
else:
    chunk_docs = []
    for doc, blocks in tqdm(doc_blocks.items(), desc="Chunking"):
        groups = semantic_chunks(blocks)
        chunk_docs += [to_document(g, doc, i) for i, g in enumerate(groups)]

    with CHUNKS_PATH.open("w", encoding="utf-8") as f:
        for d in chunk_docs:
            f.write(json.dumps({"page_content": d.page_content, "metadata": d.metadata}) + "\n")
    print(f"Saved {len(chunk_docs)} chunks to {CHUNKS_PATH}")

ids = [d.metadata["chunk_id"] for d in chunk_docs]
assert len(ids) == len(set(ids)), "duplicate chunk_ids!"

stats = pd.DataFrame([d.metadata for d in chunk_docs])
display(stats.groupby("doc")["n_words"].describe()[["count", "mean", "min", "50%", "max"]].round(0))

In [ ]:
# eyeball a few random chunks
for d in pd.Series(chunk_docs).sample(3, random_state=1):
    m = d.metadata
    print(f"--- {m['chunk_id']} | pages {m['page_start']}-{m['page_end']} | {m['n_words']} words")
    print(d.page_content[:600], "\n")

## Step 4: Index into Chroma (dense) and BM25 (sparse)

- **Chroma** is persisted to `data/index/chroma`. It rebuilds only if the set of chunk IDs changed; otherwise it reuses the existing index.
- **BM25** is built in memory, which takes seconds. The tokenizer keeps terms like `gpt-4` and `3.5` intact instead of splitting them.

In [ ]:
vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

existing_ids = set(vectorstore.get(include=[])["ids"])
if existing_ids != set(ids):
    print(f"Index out of date ({len(existing_ids)} stored vs {len(ids)} chunks), rebuilding...")
    vectorstore.reset_collection()
    BATCH = 256
    for i in tqdm(range(0, len(chunk_docs), BATCH), desc="Embedding into Chroma"):
        vectorstore.add_documents(chunk_docs[i:i + BATCH], ids=ids[i:i + BATCH])
else:
    print("Chroma index up to date, reusing it")
print("Chroma chunks:", len(vectorstore.get(include=[])["ids"]))


TOKEN_RE = re.compile(r"[a-z0-9]+(?:[-.][a-z0-9]+)*")   # keeps "gpt-4", "3.5", "bge-base"

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

bm25 = BM25Okapi([tokenize(d.page_content) for d in chunk_docs])
print("BM25 ready")


def dense_search(query, k=DENSE_K):
    return vectorstore.similarity_search(query, k=k)


def bm25_search(query, k=BM25_K):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [chunk_docs[i] for i in top if scores[i] > 0]

## Step 5: RRF fusion and reranking

**RRF (Reciprocal Rank Fusion):** each list gives a chunk `1 / (60 + rank)`. A chunk ranked high in *both* lists wins.
It only uses ranks, so it doesn't matter that BM25 scores and cosine scores are on different scales.
`rrf()` takes any number of lists, so later the GraphRAG channel just becomes a third list.

**Reranker:** the cross-encoder reads the query and chunk *together*, which is slower but much more accurate than embeddings.
So it only scores the top `RERANK_TOP_N` fused chunks.

In [ ]:
def rrf(ranked_lists, k=RRF_K):
    scores, by_id = {}, {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked, start=1):
            cid = doc.metadata["chunk_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
            by_id[cid] = doc
    return [by_id[cid] for cid in sorted(scores, key=lambda cid: scores[cid], reverse=True)]


reranker = CrossEncoder(RERANK_MODEL, device=DEVICE, max_length=512)

def rerank(query, docs, top_n=FINAL_K):
    if not docs:
        return []
    scores = reranker.predict([(query, d.page_content) for d in docs], batch_size=16)
    order = np.argsort(scores)[::-1][:top_n]
    return [(docs[i], float(scores[i])) for i in order]


def retrieve(query, debug=False):
    dense = dense_search(query)
    sparse = bm25_search(query)
    fused = rrf([dense, sparse])[:RERANK_TOP_N]
    results = rerank(query, fused)
    if not debug:
        return results

    rank_in = lambda lst, cid: next((i for i, d in enumerate(lst, 1) if d.metadata["chunk_id"] == cid), None)
    return pd.DataFrame([{
        "rerank_score": round(score, 3),
        "doc": d.metadata["doc"],
        "section": d.metadata["section"][:50],
        "pages": f"{d.metadata['page_start']}-{d.metadata['page_end']}",
        "dense_rank": rank_in(dense, d.metadata["chunk_id"]),
        "bm25_rank": rank_in(sparse, d.metadata["chunk_id"]),
        "rrf_rank": rank_in(fused, d.metadata["chunk_id"]),
        "preview": d.page_content.split("\n\n", 1)[-1][:120],
    } for d, score in results])

In [ ]:
test_queries = [
    "How does HELM measure calibration of language models?",
    "What did the GPT-4 technical report say about predictable scaling?",
    "What is homogenization in foundation models and why is it risky?",
    "How did the Sparks of AGI paper test GPT-4 on theory of mind?",
]

for q in test_queries:
    print("Q:", q)
    display(retrieve(q, debug=True))